In [1]:
# import sys
# package_path = "/localscratch/mlobo6/spadecoder/datasets/"
# if package_path not in sys.path:
#     sys.path.append(package_path)
# from spadecoder import *

## visualize spots + pie plots of results 


import pickle


import scanpy as sc

import pandas as pd



import matplotlib.pyplot as plt
from matplotlib import colors
# color_map
# sc.settings.set_figure_params(dpi=120)

plt.rcParams['figure.figsize']=(8,8) #rescale figures
# sc.settings.verbosity = 3

sc.set_figure_params(scanpy=True, dpi_save=400,dpi=150)

# plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42

import os 


import numpy as np 
import sys


In [2]:
figdir = '../fig2_a_main_low_nswaps2/'

In [3]:

def eval_perspot(gnd_truth,deconv_res):
  
    gnd_truth = gnd_truth.T
    
    deconv_res.columns = gnd_truth.columns # keeping cell ids consistent 
    assert set(deconv_res.index) == set(gnd_truth.index), "the ground truth and deconv results have different cell types"
    deconv_res = deconv_res.loc[gnd_truth.index,]

    # pearson corr 
    pearson_cor = gnd_truth.corrwith(deconv_res, axis = 0, method='pearson') # if not doing rank correlation, normalization will matter 
    # avg_corr_pe = pearson_cor.mean()


    # dom ctype - True / False 
    pred_dom_ct = deconv_res.idxmax()
    gt_dom_ct = gnd_truth.idxmax()
    is_correct_dom = (pred_dom_ct == gt_dom_ct).astype(int)
    # is_correct_dom


    # correlation between cell-types 
    correlations_pe = pd.DataFrame(index=deconv_res.T.columns, columns=gnd_truth.T.columns)
    for col1 in deconv_res.T.columns:
        for col2 in gnd_truth.T.columns:
            # gives nan when all 0's due to cell type not present in ref or query
            # correlations_sp.at[col1, col2] = deconv_res.T[col1].corr(gnd_truth.T[col2], method='spearman')
            correlations_pe.at[col1, col2] = deconv_res.T[col1].corr(gnd_truth.T[col2], method='pearson')
    # correlations_pe



    # euclidean dist 
    gnd_truth_norm = np.array(gnd_truth/gnd_truth.sum())    
    deconv_res = np.array(deconv_res/deconv_res.sum())
    # orig_rmse = np.sqrt(((gnd_truth_norm - deconv_res) ** 2).sum())
    cell_rmse = np.sqrt(((gnd_truth_norm - deconv_res) ** 2).sum(axis=0))



    per_cell_metrics = pd.DataFrame(index=gnd_truth.columns,columns=['pearson_cor','correct_dom','cell_rmse'])

    per_cell_metrics['pearson_cor'] = pearson_cor
    per_cell_metrics['correct_dom'] = is_correct_dom
    per_cell_metrics['cell_rmse'] = cell_rmse

    return per_cell_metrics, correlations_pe


In [4]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

def viz_slice_spots_metrics(spacoord_x, spacoord_y, plot_df, metric, legend=True, spot_sz=50, cmap='viridis'):
    """
    Visualize a grid of scatter subplots.
    
    Each subplot uses the same x and y coordinates (spacoord_x, spacoord_y) but 
    colors the points according to one column in plot_df. All subplots share a common colorbar.

    Parameters:
      spacoord_x : array-like
          x coordinates common to all scatter plots.
      spacoord_y : array-like
          y coordinates common to all scatter plots.
      plot_df : pandas.DataFrame
          Each column represents values for coloring the points in one subplot.
      legend : bool, default True
          If True, adds a common colorbar to the figure.
      cmap : str, default 'viridis'
          Colormap to use for the scatter plots.
    
    Returns:
      fig : matplotlib.figure.Figure
          The created figure with the subplots.
    """
    # Determine number of subplots from the DataFrame columns.
    n_plots = plot_df.shape[1]
    n_rows = 2
    n_cols = math.ceil(n_plots / n_rows)
    
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    
    # Flatten the axes array for easy iteration.
    if hasattr(axs, "flatten"):
        axs = axs.flatten()
    else:
        axs = [axs]
    
    # Calculate overall min and max from plot_df to ensure consistent color scaling.
    vmin = plot_df.min().min()
    vmax = plot_df.max().max()
    
    scatter_plots = []
    for i, col in enumerate(plot_df.columns):
        ax = axs[i]
        # Create scatter plot with same x and y coordinates and color by the column values.
        sc = ax.scatter(spacoord_x, spacoord_y, c=plot_df[col].values, cmap=cmap,
                        s=150, vmin=vmin, vmax=vmax, edgecolor='black')
        ax.axis('off')
        ax.set_title(col, fontsize=16)
        # ax.set_xlabel("X", fontsize=12)
        # ax.set_ylabel("Y", fontsize=12)
        ax.set_aspect('equal')
        scatter_plots.append(sc)
    
    # Hide any extra subplots if there are more subplots than columns in plot_df.
    for j in range(n_plots, len(axs)):
        axs[j].set_visible(False)
    
    if legend:
        # Create a ScalarMappable for a common colorbar.
        norm = Normalize(vmin=vmin, vmax=vmax)
        sm = ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        # Add a common colorbar to the right side of all subplots.
        cbar = fig.colorbar(sm, ax=axs, orientation='vertical', fraction=0.02, pad=0.04)
        cbar.set_label(metric, fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    return fig


In [5]:
import math 

def viz_slice_spots(adata_dict,celltype_classes,celltype_colors,pie_size=0.01, legend=True):

    keys = list(adata_dict.keys())

    N = len(keys)
    n_rows = 2 
    n_cols = math.ceil((N+1) / n_rows)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(10 * n_cols, 8 * n_rows))
    
    if hasattr(axs, "flatten"):
        axs = axs.flatten()
    else:
        axs = [axs]

    legend_handles = None
    for j, key in enumerate(keys):
        tmp = adata_dict[key].copy()
        # if ext_obs is not None:
        #     tmp.obs = ext_obs

    
        
        # check no extras
        assert set(tmp.obs.columns) == set(celltype_classes)

        tmp.obs = tmp.obs[celltype_classes]

        assert list(tmp.obs.columns) == celltype_classes # make sure order is correct 

        ax = axs[j]

        # Plot each point as a pie chart
        for i in range(tmp.shape[0]):
            # Add a pie chart at each (x, y) position
            pie_size = 0.04  # Size of each pie chart
            wedges, _ = ax.pie(
                tmp.obs.loc[str(i),].values,
                center=(tmp.obsm['spatial'][i,0], tmp.obsm['spatial'][i,1]),
                radius=pie_size,colors=celltype_colors,
                wedgeprops={"linewidth": 0, "edgecolor": "none"} 
            )
            if legend_handles is None:
                legend_handles = wedges

        # Set axis limits
        mina = tmp.obsm['spatial'].min(axis=0)
        maxa = tmp.obsm['spatial'].max(axis=0)
        ax.set_xlim(mina[0], maxa[0])
        ax.set_ylim(mina[1]-0.06, maxa[1]+0.06)

        # Add labels and title
        #ax.set_xlabel("X-axis", fontsize=14)
        #ax.set_ylabel("Y-axis", fontsize=14)
        ax.set_title(key, fontsize=24)
        ax.set_aspect('equal')

    # add 1 plot which is the number of cells in a spot
    scatter_ax = axs[len(keys)]
    gt_key = None
    for idx, key in  enumerate(keys):
        if 'GroundTruth' in key:
            gt_key = key
            break
    if gt_key is None:
        print("HELP!!!!!")    

    x = adata_dict[gt_key].obsm['spatial'][:, 0]
    y = adata_dict[gt_key].obsm['spatial'][:, 1]

    values = adata_dict[gt_key].obs.sum(axis=1)

    scatter = scatter_ax.scatter(x, y, c=values, cmap='viridis', s=500, edgecolor='black')
    
    scatter_ax.set_title('NumCells', fontsize=24)
    
    # Set axis limits
    mina = adata_dict[gt_key].obsm['spatial'].min(axis=0)
    maxa = adata_dict[gt_key].obsm['spatial'].max(axis=0)
    scatter_ax.set_xlim(mina[0]-0.06, maxa[0]+0.06)
    scatter_ax.set_ylim(mina[1]-0.06, maxa[1]+0.06)

    # Add labels and title
    #ax.set_xlabel("X-axis", fontsize=14)
    #ax.set_ylabel("Y-axis", fontsize=14)
    
    scatter_ax.set_aspect('equal')
     
    
    for ax in axs[N+1:]:
        ax.set_visible(False)

    

    if legend:
        fig.subplots_adjust(right=0.85)
        fig.legend(
            handles=legend_handles,
            labels=celltype_classes,
            title="CellTypes",
            loc="center left",
            bbox_to_anchor=(0.9, 0.5),
            fontsize=10,
            title_fontsize=20
        )

        cbar = fig.colorbar(scatter, ax=scatter_ax, fraction=0.046, pad=0.04)
        cbar.set_label("Sum of Cells per Spot", fontsize=12)



    # Adjust aspect ratio to ensure proper scaling
    #ax.set_aspect('equal')
    plt.tight_layout(rect = [0, 0, 0.85, 1])

    # Show the plot
    # plt.show()
    return fig

In [6]:
## visualize spots + pie plots of results 


import pickle


import scanpy as sc

import pandas as pd



import matplotlib.pyplot as plt
from matplotlib import colors
# color_map
# sc.settings.set_figure_params(dpi=120)

plt.rcParams['figure.figsize']=(8,8) #rescale figures
# sc.settings.verbosity = 3

sc.set_figure_params(scanpy=True, dpi_save=400,dpi=150)

# plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42


import math 



def viz_slice_spots_one(adata, celltype_classes, celltype_colors, pie_size=0.04, title='Tissue Slice', legend=True):
    # Ensure that the required celltype columns exist in adata.obs
    

    # fig, axs = plt.subplots(1, 1, figsize=(20, 18))

    # if hasattr(axs, "flatten"):
    #     axs = axs.flatten()
    # else:
    #     axs = [axs]

    # Reorder the cell types in adata.obs
    legend_handles = None
    assert set(adata.obs.columns) == set(celltype_classes)
    adata.obs = adata.obs[celltype_classes]
    assert list(adata.obs.columns) == celltype_classes
    # ax = axs[j]
    

    # Create a plot with subplots
    N = 1  # Only one object, so N = 1
    n_rows = 1
    n_cols = 1
    fig, ax = plt.subplots(n_rows, n_cols, figsize=(20, 16))


    
    # Plot each point as a pie chart for the single Scanpy object (adata)
    for i in range(adata.obs.shape[0]):
        # Add a pie chart at each (x, y) position
        pie_size = pie_size  # Size of each pie chart
        cell_name = adata.obs.index[i]
        wedges, _ = ax.pie(
            adata.obs.loc[cell_name,].values,
            center=(adata.obsm['spatial'][i, 0], adata.obsm['spatial'][i, 1]),
            radius=pie_size, colors=celltype_colors,
            wedgeprops={"linewidth": 0, "edgecolor": "none"}
        )

    # Set axis limits based on spatial data
    mina = adata.obsm['spatial'].min(axis=0)
    maxa = adata.obsm['spatial'].max(axis=0)
    ax.set_xlim(mina[0], maxa[0])
    ax.set_ylim(mina[1] - 0.06, maxa[1] + 0.06)

    # Add title and other settings
    ax.set_title(title, fontsize=50)
    ax.set_aspect('equal')

    if legend:
        fig.subplots_adjust(right=0.85)
        fig.legend(
            handles=wedges,
            labels=celltype_classes,
            title="CellTypes",
            loc="center left",
            bbox_to_anchor=(0.9, 0.5),
            fontsize=10,
            title_fontsize=20
        )

    plt.tight_layout(rect=[0, 0, 0.85, 1])

    return fig




## Moffitt2018

In [7]:
# spa_celltype_key='cell_type'

# 'scrna_cluster_key':
#             'spa_key':'spatial',


dataset = '1'

scrna_cluster_key = "Cell class (determined from clustering of all cells)"

deconv_dir = figdir + dataset + '/deconv/'
metrics_dir = figdir + dataset + '/metrics/'

dataset = 'Moffitt2018'
dataset_full = 'dataset1_merfish_moffitt2018'
pickle_path = '../../datasets/' + dataset_full + '/results/simulations/pickles/'
N = 50
nneigh = 10
nbdswaps = 2
par_lambda_curr = 0.1
# par_eta1_curr = 10.0
kernel3d_bw_slices_curr = 8
bandwidth_curr = 0.01
n_spatial_neigh_curr = 10
mode_nbd = 'variabletranscr'
gtalign = False # if True, use GT alignment 
aligntool = 'moscot'
key_name = 'linear_0'
num_augment = '20'
mode_nbd = 'variabletranscr'
augment = True
mode = 'multislice'
sc_type = 'sc'
suffix = '_normnolog_nov2024_1k'





deconv_dir = figdir + dataset + '/deconv/'
adata_sc = sc.read('../../datasets/' + dataset_full +   "/data/scrna_ref" + suffix + ".h5ad")

cell_type = scrna_cluster_key
celltype_colors = list(adata_sc.uns[cell_type + '_colors'])
celltype_classes  = list(adata_sc.obs[cell_type].cat.categories)


adata_spa_path = pickle_path + 'multi_slice_simulated_' + 'sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + suffix + '.pickle' # '_old.pickle'        
# read spatial file 
if not os.path.exists(adata_spa_path):
    print(adata_spa_path)
    sys.exit() 
with open(adata_spa_path, 'rb') as handle:
    adata_spa = pickle.load(handle)


key_name = 'linear_0'

result_metric = ['orig_rmse',   'avg_corr_pe','avg_jsd']


all_algos = {}

partesting_str =     '_partesting_' + 'parlambda_' + str(par_lambda_curr) +   '_parbw_' + str(bandwidth_curr) +  '_nbdtype_' + mode_nbd + '_gtalign_' + str(gtalign) + '_augment_' + str(augment) + '_kernel3d_bw_slices_' + str(kernel3d_bw_slices_curr) + '_num_augment_' + str(num_augment) +  '_realalign_' + str(aligntool) 
res_file = 'deconv_' + mode + '_' + mode_nbd + '_sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + partesting_str +'_sc_sim.pickle'
print(res_file)
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_sc_ms = pickle.load(handle)
all_algos['SpaDecoder'] = metrics_sc_ms

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_CARD.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_card = pickle.load(handle)
metrics_card = metrics_card['linear_0']
all_algos['CARD'] = metrics_card

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_cell2location.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_cell2loc = pickle.load(handle)
metrics_cell2loc = metrics_cell2loc['linear_0']
all_algos['Cell2location'] = metrics_cell2loc

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_tangram.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_tangram = pickle.load(handle)
metrics_tangram = metrics_tangram['linear_0']
all_algos['Tangram'] = metrics_tangram

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_tangramsc.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_tangramsc = pickle.load(handle)
metrics_tangramsc = metrics_tangramsc['linear_0']
all_algos['Tangramsc'] = metrics_tangramsc








deconv_multislice_variabletranscr_sptsz_50_nneigh_10_nbdswaps_2_partesting_parlambda_0.1_parbw_0.01_nbdtype_variabletranscr_gtalign_False_augment_True_kernel3d_bw_slices_8_num_augment_20_realalign_moscot_sc_sim.pickle


/tmp/ipykernel_240121/915490516.py:54: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  adata_spa = pickle.load(handle)
/tmp/ipykernel_240121/915490516.py:68: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is t

In [8]:
all_algos['GroundTruth'] = adata_spa.copy()

In [10]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

key_name = 'linear_0'



#     with PdfPages(output_pdf) as pdf:
metrics_dict = {}
corr_dict = {}


for realidx in all_algos['SpaDecoder'].keys(): # real slice 
    for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
        adata_dict = {}
        gnd_truth = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
        tmp_set = set(celltype_classes) - set(gnd_truth.obs.columns)

        if len(tmp_set) != 0:
            for entry_tmp in tmp_set:
                gnd_truth.obs[entry_tmp] = 0.0
                gnd_truth.obs = gnd_truth.obs[celltype_classes]

        for algo_name in all_algos:
            if algo_name != 'GroundTruth':
                deconv_res = all_algos[algo_name][realidx][simidx].copy()
                samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                metrics_dict[samp_name], corr_dict[samp_name] = eval_perspot(gnd_truth.obs,deconv_res)



                

/project/mlobo6/miniconda3/envs/cell2location/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/project/mlobo6/miniconda3/envs/cell2location/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/project/mlobo6/miniconda3/envs/cell2location/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/project/mlobo6/miniconda3/envs/cell2location/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/project/mlobo6/miniconda3/envs/cell2location/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/project/mlobo6/miniconda3/envs/cell2location/lib/pytho

In [31]:
## run viz code 
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


key_name = 'linear_0'

for metric in ['pearson_cor','correct_dom','cell_rmse']:

    output_pdf = figdir + "./viz_slices_deconv_output_dataset_" + dataset + "_metric_" + metric + ".pdf"


    with PdfPages(output_pdf) as pdf:

        for realidx in all_algos["SpaDecoder"].keys(): # real slice 
            for simidx in all_algos["SpaDecoder"][realidx].keys(): # sim slice 
                plot_df = pd.DataFrame(index=all_algos['GroundTruth'][key_name][simidx][realidx].obs.index)
                spacoord_x = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,0]
                spacoord_y = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,1]
                for algo_name in all_algos:
                    if algo_name != 'GroundTruth':
                        samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                        plot_df[samp_name] = metrics_dict[samp_name][metric]
 
                        # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
                fig = viz_slice_spots_metrics(spacoord_x,spacoord_y, plot_df,metric,spot_sz=150, legend=True)

                # fig = plt.gcf()

                pdf.savefig(fig)
                
                
                # plt.show()
                plt.close(fig)
                

                

/tmp/ipykernel_2484007/30046532.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/30046532.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/30046532.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/30046532.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/30046532.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/30046532.py:74: U

In [10]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

with PdfPages(output_pdf) as pdf:

    for realidx in all_algos['SpaDecoder'].keys(): # real slice 
        for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
            adata_dict = {}
            for algo_name in all_algos:
                if algo_name != 'GroundTruth':
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
                else:
                    if len(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns) < len(celltype_classes):
                        to_add = list(set(celltype_classes) - set(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns)) 
                        all_algos['GroundTruth'][key_name][simidx][realidx].obs[to_add] = 0.0
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors,pie_size=0.04, legend=False)

            

            # fig = plt.gcf()

            pdf.savefig(fig)
            
            # plt.show()
            plt.close(fig)
            
        #     break
        # break
    
                

In [11]:


from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_all_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

# with PdfPages(output_pdf) as pdf:

for realidx in ['3']:#all_algos['SpaDecoder'].keys(): # real slice 
    for simidx in [0]:# all_algos['SpaDecoder'][realidx].keys(): # sim slice 
        adata_dict = {}
        for algo_name in all_algos:
            if algo_name != 'GroundTruth':
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
            else:
                if len(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns) < len(celltype_classes):
                    to_add = list(set(celltype_classes) - set(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns)) 
                    all_algos['GroundTruth'][key_name][simidx][realidx].obs[to_add] = 0.0
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            # fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors,pie_size=0.04, legend=False)

            fig = viz_slice_spots_one(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)],celltype_classes,celltype_colors,legend=False,pie_size=0.04, title=algo_name + ' Sample ' + str(realidx) + ' Slice ' + str(simidx))

        
            fig_path = os.path.join(figdir, "viz_slices_deconv_all_output_dataset_" + dataset + '_'  + algo_name + 'Sample_' + str(realidx) + 'Sim_' + str(simidx) + ".pdf")
            fig.savefig(fig_path, dpi=1200, bbox_inches='tight')
            plt.close(fig)


            # pdf.savefig(fig)
        
            # # plt.show()
            # plt.close(fig)

                # fig = plt.gcf()

                
            
        #     break
        # break
    
                

In [12]:
figdir

'../fig2_a_main_low_nswaps2/'

## Choi2023

In [13]:
# spa_celltype_key='cell_type'

# 'scrna_cluster_key':
#             'spa_key':'spatial',


dataset = '2'

scrna_cluster_key = "majorclass"

deconv_dir = figdir + dataset + '/deconv/'
metrics_dir = figdir + dataset + '/metrics/'

dataset = 'Choi2023'
dataset_full = 'dataset2_merfish_retina_2022'
pickle_path = '../../datasets/' + dataset_full + '/results/simulations/pickles/'
N = 50
nneigh = 10
nbdswaps = 2
par_lambda_curr = 0.1
# par_eta1_curr = 10.0
kernel3d_bw_slices_curr = 8
bandwidth_curr = 0.01
n_spatial_neigh_curr = 10
mode_nbd = 'variabletranscr'
gtalign = False # if True, use GT alignment 
aligntool = 'moscot'
key_name = 'linear_0'
num_augment = '20'
mode_nbd = 'variabletranscr'
augment = True
mode = 'multislice'
sc_type = 'sc'
suffix = '_norm1knolog'





deconv_dir = figdir + dataset + '/deconv/'
adata_sc = sc.read('../../datasets/' + dataset_full +   "/data/scrna_ref_norm1knolog.h5ad")

cell_type = scrna_cluster_key
celltype_colors = list(adata_sc.uns[cell_type + '_colors'])
celltype_classes  = list(adata_sc.obs[cell_type].cat.categories)


adata_spa_path = pickle_path + 'multi_slice_simulated_' + 'sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + suffix + '.pickle' # '_old.pickle'        
# read spatial file 
if not os.path.exists(adata_spa_path):
    sys.exit() 
with open(adata_spa_path, 'rb') as handle:
    adata_spa = pickle.load(handle)


key_name = 'linear_0'

result_metric = ['orig_rmse',   'avg_corr_pe','avg_jsd']


all_algos = {}

partesting_str =     '_partesting_' + 'parlambda_' + str(par_lambda_curr) +   '_parbw_' + str(bandwidth_curr) +  '_nbdtype_' + mode_nbd + '_gtalign_' + str(gtalign) + '_augment_' + str(augment) + '_kernel3d_bw_slices_' + str(kernel3d_bw_slices_curr) + '_num_augment_' + str(num_augment) +  '_realalign_' + str(aligntool) 
res_file = 'deconv_' + mode + '_' + mode_nbd + '_sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + partesting_str +'_sc_sim.pickle'
print(res_file)
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_sc_ms = pickle.load(handle)
all_algos['SpaDecoder'] = metrics_sc_ms

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_CARD.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_card = pickle.load(handle)
metrics_card = metrics_card['linear_0']
all_algos['CARD'] = metrics_card

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_cell2location.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_cell2loc = pickle.load(handle)
metrics_cell2loc = metrics_cell2loc['linear_0']
all_algos['Cell2location'] = metrics_cell2loc

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_tangram.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_tangram = pickle.load(handle)
metrics_tangram = metrics_tangram['linear_0']
all_algos['Tangram'] = metrics_tangram

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_tangramsc.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_tangramsc = pickle.load(handle)
metrics_tangramsc = metrics_tangramsc['linear_0']
all_algos['Tangramsc'] = metrics_tangramsc








/tmp/ipykernel_240121/4086419632.py:53: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  adata_spa = pickle.load(handle)
/tmp/ipykernel_240121/4086419632.py:67: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is

deconv_multislice_variabletranscr_sptsz_50_nneigh_10_nbdswaps_2_partesting_parlambda_0.1_parbw_0.01_nbdtype_variabletranscr_gtalign_False_augment_True_kernel3d_bw_slices_8_num_augment_20_realalign_moscot_sc_sim.pickle


/tmp/ipykernel_240121/4086419632.py:84: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  metrics_tangram = pickle.load(handle)
/tmp/ipykernel_240121/4086419632.py:90: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If t

In [14]:
all_algos['GroundTruth'] = adata_spa.copy()

In [15]:
samples_to_keep = ['20.0', '21.0', '22.0', '23.0', '24.0', '25.0', '26.0', '40.0', '41.0', '42.0', '43.0', '44.0', '45.0', '46.0', '50.0', '51.0', '52.0', '53.0', '54.0', '55.0', '56.0', '10.0','14.0','15.0','16.0']


for entry in all_algos['GroundTruth']['linear_0'].keys():
    for entry2 in samples_to_keep:
        tmpa = all_algos['GroundTruth']['linear_0'][entry][entry2].obs.sum()
        if tmpa.min() < 40.0:
            print(entry, entry2)
        # if 'Cone' in tmpa.index:
        #     if tmpa['Cone'] <= 5.0:
        #         print(entry, entry2)
        #         # continue
        # else:
        #     print(entry, entry2)

0 40.0
0 52.0
0 53.0
0 56.0
1 40.0
1 52.0
1 53.0
1 56.0
2 40.0
2 52.0
2 53.0
2 56.0
3 40.0
3 52.0
3 53.0
3 56.0
4 40.0
4 52.0
4 53.0
4 56.0
5 40.0
5 52.0
5 53.0
5 56.0
6 40.0
6 52.0
6 53.0
6 56.0
7 40.0
7 52.0
7 53.0
7 56.0
8 40.0
8 52.0
8 53.0
8 56.0
9 40.0
9 52.0
9 53.0
9 56.0
10 40.0
10 52.0
10 53.0
10 56.0


In [17]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

key_name = 'linear_0'



#     with PdfPages(output_pdf) as pdf:
metrics_dict = {}
corr_dict = {}


for realidx in all_algos['SpaDecoder'].keys(): # real slice 
    for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
        adata_dict = {}
        gnd_truth = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
        tmp_set = set(celltype_classes) - set(gnd_truth.obs.columns)

        if len(tmp_set) != 0:
            for entry_tmp in tmp_set:
                gnd_truth.obs[entry_tmp] = 0.0
                gnd_truth.obs = gnd_truth.obs[celltype_classes]

        for algo_name in all_algos:
            if algo_name != 'GroundTruth':
                deconv_res = all_algos[algo_name][realidx][simidx].copy()
                samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                metrics_dict[samp_name], corr_dict[samp_name] = eval_perspot(gnd_truth.obs,deconv_res)




In [14]:

                
## run viz code 
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


key_name = 'linear_0'

for metric in ['pearson_cor','correct_dom','cell_rmse']:

    output_pdf = figdir + "./viz_slices_deconv_output_dataset_" + dataset + "_metric_" + metric + ".pdf"


    with PdfPages(output_pdf) as pdf:

        for realidx in all_algos["SpaDecoder"].keys(): # real slice 
            for simidx in all_algos["SpaDecoder"][realidx].keys(): # sim slice 
                plot_df = pd.DataFrame(index=all_algos['GroundTruth'][key_name][simidx][realidx].obs.index)
                spacoord_x = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,0]
                spacoord_y = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,1]
                for algo_name in all_algos:
                    if algo_name != 'GroundTruth':
                        samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                        plot_df[samp_name] = metrics_dict[samp_name][metric]
 
                        # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
                fig = viz_slice_spots_metrics(spacoord_x,spacoord_y, plot_df,metric,spot_sz=150, legend=True)

                # fig = plt.gcf()

                pdf.savefig(fig)
                
                
                # plt.show()
                plt.close(fig)
                


/tmp/ipykernel_2825034/971990981.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2825034/971990981.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2825034/971990981.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2825034/971990981.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2825034/971990981.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2825034/971990981.py

In [ ]:

                
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

with PdfPages(output_pdf) as pdf:

    for realidx in all_algos['SpaDecoder'].keys(): # real slice 
        for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
            adata_dict = {}
            for algo_name in all_algos:
                if algo_name != 'GroundTruth':
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
                else:
                    if len(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns) < len(celltype_classes):
                        to_add = list(set(celltype_classes) - set(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns)) 
                        all_algos['GroundTruth'][key_name][simidx][realidx].obs[to_add] = 0.0
                    
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors, pie_size=0.04, legend=False)

            

            # fig = plt.gcf()

            pdf.savefig(fig)
            
            # plt.show()
            plt.close(fig)
            
        #     break
        # break
    
                

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_all_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

with PdfPages(output_pdf) as pdf:

    for realidx in all_algos['SpaDecoder'].keys(): # real slice 
        for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
            adata_dict = {}
            for algo_name in all_algos:
                if algo_name != 'GroundTruth':
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
                else:
                    if len(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns) < len(celltype_classes):
                        to_add = list(set(celltype_classes) - set(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns)) 
                        all_algos['GroundTruth'][key_name][simidx][realidx].obs[to_add] = 0.0
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
                # fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors,pie_size=0.04, legend=False)

                fig = viz_slice_spots_one(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)],celltype_classes,celltype_colors,legend=False,pie_size=0.04, title=algo_name + ':' + 'Sample ' + str(realidx) + ' Slice ' + str(simidx))

            

                pdf.savefig(fig)
            
                # plt.show()
                plt.close(fig)

                # fig = plt.gcf()

                
            
        #     break
        # break
    
                

In [18]:


from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_all_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

# with PdfPages(output_pdf) as pdf:

for realidx in ['40.0']:#all_algos['SpaDecoder'].keys(): # real slice 
    for simidx in [0]:# all_algos['SpaDecoder'][realidx].keys(): # sim slice 
        adata_dict = {}
        for algo_name in all_algos:
            if algo_name != 'GroundTruth':
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
            else:
                if len(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns) < len(celltype_classes):
                    to_add = list(set(celltype_classes) - set(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns)) 
                    all_algos['GroundTruth'][key_name][simidx][realidx].obs[to_add] = 0.0
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            # fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors,pie_size=0.04, legend=False)

            fig = viz_slice_spots_one(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)],celltype_classes,celltype_colors,legend=False,pie_size=0.03, title=algo_name + ' Sample ' + str(realidx) + ' Slice ' + str(simidx))

        
            fig_path = os.path.join(figdir, "viz_slices_deconv_all_output_dataset_" + dataset + '_'  + algo_name + 'Sample_' + str(realidx) + 'Sim_' + str(simidx) + ".pdf")
            fig.savefig(fig_path, dpi=1200, bbox_inches='tight')
            plt.close(fig)


            # pdf.savefig(fig)
        
            # # plt.show()
            # plt.close(fig)

                # fig = plt.gcf()

                
            
        #     break
        # break
    
                

## Haviv 2025

In [19]:
spa_celltype_key='cell_type'

dataset = '9'

scrna_cluster_key = "cell_type"

deconv_dir = figdir + dataset + '/deconv/'
metrics_dir = figdir + dataset + '/metrics/'

dataset = 'Haviv2025'
dataset_full = 'dataset9_xenuim'
pickle_path = '../../datasets/' + dataset_full + '/results/simulations/pickles/'
N = 50
nneigh = 10
nbdswaps = 2
par_lambda_curr = 0.1
# par_eta1_curr = 10.0
kernel3d_bw_slices_curr = 8
bandwidth_curr = 0.01
n_spatial_neigh_curr = 10
mode_nbd = 'variabletranscr'
gtalign = False # if True, use GT alignment 
aligntool = 'moscot'
key_name = 'linear_0'
num_augment = '20'
mode_nbd = 'variabletranscr'
augment = True
mode = 'multislice'
sc_type = 'sc'
suffix = '_norm1knolog'



celltype_classes = ['Astrocyte',
 'Endothelial',
 'Excitatory Neuron',
 'Fibroblast',
 'Immune',
 'Inhibitory Neuron',
 'Oligodendrocyte',
 'Tumor']

celltype_colors = ['#1f77b4',
 '#ff7f0e',
 '#279e68',
 '#d62728',
 '#aa40fc',
 '#8c564b',
 '#e377c2',
 '#b5bd61']


deconv_dir = figdir + dataset + '/deconv/'
adata_sc = sc.read('../../datasets/' + dataset_full +   "/data/scrna_ref_norm1knolog.h5ad")


adata_spa_path = pickle_path + 'multi_slice_simulated_' + 'sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + suffix + '.pickle' # '_old.pickle'        
# read spatial file 
if not os.path.exists(adata_spa_path):
    sys.exit() 
with open(adata_spa_path, 'rb') as handle:
    adata_spa = pickle.load(handle)


key_name = 'linear_0'

result_metric = ['orig_rmse',   'avg_corr_pe','avg_jsd']


all_algos = {}

partesting_str =     '_partesting_' + 'parlambda_' + str(par_lambda_curr) +   '_parbw_' + str(bandwidth_curr) +  '_nbdtype_' + mode_nbd + '_gtalign_' + str(gtalign) + '_' + str(augment) + '_kernel3d_bw_slices_' + str(kernel3d_bw_slices_curr) + '_num_augment_' + str(num_augment) +  '_realalign_' + str(aligntool) 
res_file = 'deconv_' + mode + '_' + mode_nbd + '_sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + partesting_str +'_sc_sim.pickle'
print(res_file)
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_sc_ms = pickle.load(handle)
all_algos['SpaDecoder'] = metrics_sc_ms

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_CARD.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_card = pickle.load(handle)
metrics_card = metrics_card['linear_0']
all_algos['CARD'] = metrics_card

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_cell2location.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_cell2loc = pickle.load(handle)
metrics_cell2loc = metrics_cell2loc['linear_0']
all_algos['Cell2location'] = metrics_cell2loc

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_tangram.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_tangram = pickle.load(handle)
metrics_tangram = metrics_tangram['linear_0']
all_algos['Tangram'] = metrics_tangram

res_file = 'deconv_st1_sptsz_' + str(N) + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + '_tangramsc.pickle'
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_tangramsc = pickle.load(handle)
metrics_tangramsc = metrics_tangramsc['linear_0']
all_algos['Tangramsc'] = metrics_tangramsc








deconv_multislice_variabletranscr_sptsz_50_nneigh_10_nbdswaps_2_partesting_parlambda_0.1_parbw_0.01_nbdtype_variabletranscr_gtalign_False_True_kernel3d_bw_slices_8_num_augment_20_realalign_moscot_sc_sim.pickle


/tmp/ipykernel_240121/1596352912.py:62: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  adata_spa = pickle.load(handle)
/tmp/ipykernel_240121/1596352912.py:76: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is

In [20]:
all_algos['GroundTruth'] = adata_spa.copy()

In [22]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

key_name = 'linear_0'



#     with PdfPages(output_pdf) as pdf:
metrics_dict = {}
corr_dict = {}


for realidx in all_algos['SpaDecoder'].keys(): # real slice 
    for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
        adata_dict = {}
        gnd_truth = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
        tmp_set = set(celltype_classes) - set(gnd_truth.obs.columns)

        if len(tmp_set) != 0:
            for entry_tmp in tmp_set:
                gnd_truth.obs[entry_tmp] = 0.0
                gnd_truth.obs = gnd_truth.obs[celltype_classes]

        for algo_name in all_algos:
            if algo_name != 'GroundTruth':
                deconv_res = all_algos[algo_name][realidx][simidx].copy()
                samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                metrics_dict[samp_name], corr_dict[samp_name] = eval_perspot(gnd_truth.obs,deconv_res)



                

In [ ]:
## run viz code 
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


key_name = 'linear_0'

for metric in ['pearson_cor','correct_dom','cell_rmse']:

    output_pdf = figdir + "./viz_slices_deconv_output_dataset_" + dataset + "_metric_" + metric + ".pdf"


    with PdfPages(output_pdf) as pdf:

        for realidx in all_algos["SpaDecoder"].keys(): # real slice 
            for simidx in all_algos["SpaDecoder"][realidx].keys(): # sim slice 
                plot_df = pd.DataFrame(index=all_algos['GroundTruth'][key_name][simidx][realidx].obs.index)
                spacoord_x = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,0]
                spacoord_y = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,1]
                for algo_name in all_algos:
                    if algo_name != 'GroundTruth':
                        samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                        plot_df[samp_name] = metrics_dict[samp_name][metric]
 
                        # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
                fig = viz_slice_spots_metrics(spacoord_x,spacoord_y, plot_df,metric,spot_sz=50, legend=True)

                # fig = plt.gcf()

                pdf.savefig(fig)
                
                
                # plt.show()
                plt.close(fig)
                

                

/tmp/ipykernel_2484007/3614488119.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/3614488119.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/3614488119.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/3614488119.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/3614488119.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
/tmp/ipykernel_2484007/3614488

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

with PdfPages(output_pdf) as pdf:

    for realidx in all_algos['SpaDecoder'].keys(): # real slice 
        for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
            adata_dict = {}
            for algo_name in all_algos:
                if algo_name != 'GroundTruth':
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
                else:
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors, pie_size=0.01, legend=False)

            

            # fig = plt.gcf()

            pdf.savefig(fig)
            
            # plt.show()
            plt.close(fig)
            
        #     break
        # break
    
                

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_all_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

with PdfPages(output_pdf) as pdf:

    for realidx in all_algos['SpaDecoder'].keys(): # real slice 
        for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
            adata_dict = {}
            for algo_name in all_algos:
                if algo_name != 'GroundTruth':
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
                else:
                    if len(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns) < len(celltype_classes):
                        to_add = list(set(celltype_classes) - set(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns)) 
                        all_algos['GroundTruth'][key_name][simidx][realidx].obs[to_add] = 0.0
                    adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                    # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
                # fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors,pie_size=0.04, legend=False)

                fig = viz_slice_spots_one(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)],celltype_classes,celltype_colors,legend=False,pie_size=0.01, title=algo_name + ':' + 'Sample ' + str(realidx) + ' Slice ' + str(simidx))

            

                pdf.savefig(fig)
            
                # plt.show()
                plt.close(fig)

                # fig = plt.gcf()

                
            
        #     break
        # break
    
                

In [24]:


from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


output_pdf = figdir + "/viz_slices_deconv_all_output_dataset_" + dataset + ".pdf"
key_name = 'linear_0'

# with PdfPages(output_pdf) as pdf:

for realidx in ['0']:#all_algos['SpaDecoder'].keys(): # real slice 
    for simidx in [0]:# all_algos['SpaDecoder'][realidx].keys(): # sim slice 
        adata_dict = {}
        for algo_name in all_algos:
            if algo_name != 'GroundTruth':
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)].obs = all_algos[algo_name][realidx][simidx].T
            else:
                if len(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns) < len(celltype_classes):
                    to_add = list(set(celltype_classes) - set(all_algos['GroundTruth'][key_name][simidx][realidx].obs.columns)) 
                    all_algos['GroundTruth'][key_name][simidx][realidx].obs[to_add] = 0.0
                adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)] = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
                # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            # fig = viz_slice_spots(adata_dict,celltype_classes,celltype_colors,pie_size=0.04, legend=False)

            fig = viz_slice_spots_one(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)],celltype_classes,celltype_colors,legend=False,pie_size=0.01, title=algo_name + ' Sample ' + str(realidx) + ' Slice ' + str(simidx))

        
            fig_path = os.path.join(figdir, "viz_slices_deconv_all_output_dataset_" + dataset + '_'  + algo_name + 'Sample_' + str(realidx) + 'Sim_' + str(simidx) + ".pdf")
            fig.savefig(fig_path, dpi=1200, bbox_inches='tight')
            plt.close(fig)


            # pdf.savefig(fig)
        
            # # plt.show()
            # plt.close(fig)

                # fig = plt.gcf()

                
            
        #     break
        # break
    
                